# Data Cleaning & Feature Engineering

## Project

E-Commerce Sales Intelligence Platform

### Objective

This notebook cleans, validates, and prepares the Olist Brazilian E-Commerce dataset for SQL modeling, business analysis, and Power BI dashboards.

Prepared By: Tsering Gurung

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
data_path = Path("../data/raw")

datasets = {}

for file in sorted(data_path.glob("*.csv")):
    datasets[file.stem] = pd.read_csv(file)

print(f"Loaded {len(datasets)} datasets.")

Loaded 9 datasets.


In [3]:
customers = datasets["olist_customers_dataset"]
orders = datasets["olist_orders_dataset"]
items = datasets["olist_order_items_dataset"]
payments = datasets["olist_order_payments_dataset"]
products = datasets["olist_products_dataset"]
reviews = datasets["olist_order_reviews_dataset"]
sellers = datasets["olist_sellers_dataset"]
geolocation = datasets["olist_geolocation_dataset"]
translations = datasets["product_category_name_translation"]

In [4]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [5]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

In [6]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


# Delivery_days
We created this feature to understand how long does it take Olist to deliver an order measured in days. This step helps us undertand the delays.

In [7]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [8]:
orders["delivery_days"].describe()

count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64

In [9]:
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

In [10]:
orders["is_late_delivery"] = (
    orders["delivery_delay_days"] > 0
)

# Relationship Validation

## Objective

Validate that primary and foreign key relationships are consistent across the datasets before building SQL models and dashboards.

In [11]:
missing_customers = (
    ~orders["customer_id"].isin(customers["customer_id"])
).sum()

print(f"Orders without matching customer: {missing_customers}")

Orders without matching customer: 0


In [12]:
missing_orders = (
    ~items["order_id"].isin(orders["order_id"])
).sum()

print(f"Order items without matching order: {missing_orders}")

Order items without matching order: 0


In [13]:
missing_payments = (
    ~payments["order_id"].isin(orders["order_id"])
).sum()

print(f"Payments without matching order: {missing_payments}")

Payments without matching order: 0


In [14]:
missing_reviews = (
    ~reviews["order_id"].isin(orders["order_id"])
).sum()

print(f"Reviews without matching order: {missing_reviews}")

Reviews without matching order: 0


In [15]:
missing_sellers = (
    ~items["seller_id"].isin(sellers["seller_id"])
).sum()

print(f"Unknown sellers: {missing_sellers}")

Unknown sellers: 0


In [16]:
missing_products = (
    ~items["product_id"].isin(products["product_id"])
).sum()

print(f"Unknown products: {missing_products}")

Unknown products: 0


In [17]:
validation_df = pd.DataFrame({
    "Relationship": [
        "Orders → Customers",
        "Items → Orders",
        "Payments → Orders",
        "Reviews → Orders",
        "Items → Sellers",
        "Items → Products"
    ],
    "Missing Records": [
        missing_customers,
        missing_orders,
        missing_payments,
        missing_reviews,
        missing_sellers,
        missing_products
    ]
})

validation_df

,Relationship,Missing Records
0,Orders → Customers,0
1,Items → Orders,0
2,Payments → Orders,0
3,Reviews → Orders,0
4,Items → Sellers,0
5,Items → Products,0


# Relationship Validation Summary

## Findings

All major relationships between the transactional and dimension datasets were successfully validated. No orphaned records were identified, indicating there is a strong referential integrity across the source data.

This provides confidence that the datasets can be joined reliably for SQL modeling, business analysis, and dashboard development.

# Feature Engineering

## Objective

Create additional variables that make business analysis easier and improve downstream SQL queries, dashboards, and machine learning models.

In [18]:
orders["order_year"] = (
    orders["order_purchase_timestamp"]
    .dt.year
)

In [19]:
orders["order_month"] = (
    orders["order_purchase_timestamp"]
    .dt.month
)

In [20]:
orders["month_name"] = (
    orders["order_purchase_timestamp"]
    .dt.month_name()
)

In [21]:
orders["order_quarter"] = (
    orders["order_purchase_timestamp"]
    .dt.quarter
)

In [22]:
orders["weekday"] = (
    orders["order_purchase_timestamp"]
    .dt.day_name()
)

In [23]:
orders["is_weekend_purchase"] = (
    orders["order_purchase_timestamp"]
    .dt.weekday >= 5
)

In [24]:
orders["purchase_hour"] = (
    orders["order_purchase_timestamp"]
    .dt.hour
)

In [25]:
orders[
    [
        "order_purchase_timestamp",
        "order_year",
        "order_month",
        "month_name",
        "order_quarter",
        "weekday",
        "purchase_hour",
        "is_weekend_purchase"
    ]
].head(4)

,order_purchase_timestamp,order_year,order_month,month_name,order_quarter,weekday,purchase_hour,is_weekend_purchase
0,2017-10-02 10:56:33,2017,10,October,4,Monday,10,False
1,2018-07-24 20:41:37,2018,7,July,3,Tuesday,20,False
2,2018-08-08 08:38:49,2018,8,August,3,Wednesday,8,False
3,2017-11-18 19:28:06,2017,11,November,4,Saturday,19,True


# Feature Engineering Summary

## Business Value

The newly engineered features simplify time-based analysis and reduce repetitive calculations in downstream SQL queries and Power BI dashboards.

These variables enable analyses such as:

- Monthly revenue trends
- Quarterly performance
- Peak purchasing hours
- Weekday versus weekend purchasing behavior
- Seasonal demand patterns

# Master Analytics Table

## Objective

Create an analysis-ready dataset by combining transactional, customer, product, seller, payment, review, and delivery information.

The final table will support SQL analysis, Power BI reporting, and business decision-making.

## Table Grain

Each row in the master analytics table represents one individual item within an order.

An order containing multiple products will therefore appear across multiple rows.

In [26]:
print("Orders:", orders.shape)
print("Order items:", items.shape)
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)

Orders: (99441, 18)
Order items: (112650, 7)
Customers: (99441, 5)
Products: (32951, 9)
Sellers: (3095, 4)
Payments: (103886, 5)
Reviews: (99224, 7)


In [27]:
products_clean = products.merge(
    translations,
    on="product_category_name",
    how="left"
)

In [28]:
products_clean[
    [
        "product_id",
        "product_category_name",
        "product_category_name_english"
    ]
].head()

,product_id,product_category_name,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares


In [29]:
payment_summary = (
    payments.groupby("order_id", as_index=False)
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        payment_methods=("payment_type", "nunique")
    )
)

In [30]:
payment_summary.head()


,order_id,total_payment_value,payment_installments,payment_methods
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,1
2,000229ec398224ef6ca0657da4fc703e,216.87,5,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1


In [31]:
review_summary = (
    reviews.groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_id", "nunique")
    )
)

In [32]:
master_df = items.copy()

In [33]:
master_df = master_df.merge(
    orders,
    on="order_id",
    how="left",
    validate="many_to_one"
)

In [34]:
master_df = master_df.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

In [35]:
master_df = master_df.merge(
    products_clean,
    on="product_id",
    how="left",
    validate="many_to_one"
)

In [36]:
master_df = master_df.merge(
    sellers,
    on="seller_id",
    how="left",
    validate="many_to_one"
)

In [37]:
master_df = master_df.merge(
    payment_summary,
    on="order_id",
    how="left",
    validate="many_to_one"
)

In [38]:
master_df = master_df.merge(
    review_summary,
    on="order_id",
    how="left",
    validate="many_to_one"
)

In [39]:
master_df.shape

(112650, 45)

In [40]:
master_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,total_payment_value,payment_installments,payment_methods,review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,14.0,cool_stuff,27277,volta redonda,SP,72.19,2.0,1.0,5.0,1.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,40.0,pet_shop,3471,sao paulo,SP,259.83,3.0,1.0,4.0,1.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,33.0,furniture_decor,37564,borda da mata,MG,216.87,5.0,1.0,5.0,1.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,15.0,perfumery,14403,franca,SP,25.78,2.0,1.0,4.0,1.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,30.0,garden_tools,87900,loanda,PR,218.04,3.0,1.0,5.0,1.0


In [41]:
master_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 45 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       112650 non-null  str           
 1   order_item_id                  112650 non-null  int64         
 2   product_id                     112650 non-null  str           
 3   seller_id                      112650 non-null  str           
 4   shipping_limit_date            112650 non-null  str           
 5   price                          112650 non-null  float64       
 6   freight_value                  112650 non-null  float64       
 7   customer_id                    112650 non-null  str           
 8   order_status                   112650 non-null  str           
 9   order_purchase_timestamp       112650 non-null  datetime64[us]
 10  order_approved_at              112635 non-null  datetime64[us]
 11  order_deliv

In [42]:
master_df["item_revenue"] = (
    master_df["price"]
    + master_df["freight_value"]
)

In [43]:
master_df["freight_percentage"] = np.where(
    master_df["price"] > 0,
    master_df["freight_value"] / master_df["price"],
    np.nan
)

In [44]:
print("Order item rows:", len(items))
print("Master table rows:", len(master_df))
print("Difference:", len(master_df) - len(items))

Order item rows: 112650
Master table rows: 112650
Difference: 0


In [45]:
master_df["shipping_limit_date"] = pd.to_datetime(
    master_df["shipping_limit_date"],
    errors="coerce"
)

In [46]:
master_df[["shipping_limit_date"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 1 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   shipping_limit_date  112650 non-null  datetime64[us]
dtypes: datetime64[us](1)
memory usage: 880.2 KB


In [47]:
master_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,seller_zip_code_prefix,seller_city,seller_state,total_payment_value,payment_installments,payment_methods,review_score,review_count,item_revenue,freight_percentage
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,27277,volta redonda,SP,72.19,2.0,1.0,5.0,1.0,72.19,0.225637
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,3471,sao paulo,SP,259.83,3.0,1.0,4.0,1.0,259.83,0.083076
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,37564,borda da mata,MG,216.87,5.0,1.0,5.0,1.0,216.87,0.089799
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,14403,franca,SP,25.78,2.0,1.0,4.0,1.0,25.78,0.984604
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,87900,loanda,PR,218.04,3.0,1.0,5.0,1.0,218.04,0.090745


In [48]:
master_df[
    [
        "order_id",
        "product_id",
        "customer_id",
        "price",
        "review_score",
        "product_category_name_english"
    ]
].head()

,order_id,product_id,customer_id,price,review_score,product_category_name_english
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,3ce436f183e68e07877b285a838db11a,58.90,5.0,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,f6dd3ec061db4e3987629fe6b26e5cce,239.90,4.0,pet_shop
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,6489ae5e4333f3693df5ad4372dab6d3,199.00,5.0,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,d4eb9395c8c0431ee92fce09860c5a06,12.99,4.0,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,58dbd0b2d70206bf40e62cd34e84d795,199.90,5.0,garden_tools


In [49]:
from pathlib import Path

output_path = Path("../data/processed")
output_path.mkdir(exist_ok=True)

master_df.to_csv(
    output_path / "master_analytics_table.csv",
    index=False
)

print("Master Analytics Table saved successfully.")

Master Analytics Table saved successfully.


In [50]:
master_df.to_csv(
    "../data/processed/master_analytics_table.csv",
    index=False
)